# Unified Qwen Engine: Multi-GPU Inference & Benchmarks
### Accelerated on Kaggle Dual NVIDIA Tesla T4 GPUs (2x 16GB `sm_75`, 32 GB Total VRAM)
### Powered by Pre-Compiled Release Wheels (`v0.2.0`) & Hugging Face Hub (`@areen09`)

Welcome to the dedicated inference and benchmarking notebook for the **Unified Qwen CUDA Engine**!

In this notebook, we run bare-metal accelerated inference directly on Kaggle's dual **NVIDIA Tesla T4** GPUs. All model architectures—pure Grouped Query Attention (Qwen 2.5), Hybrid Gated DeltaNet (Qwen 3.5), and large-scale 2-way Tensor Parallelism (Qwen 3.8)—are fully supported with **W4A16 Group-128 quantization** and Turing Tensor Core acceleration.

---

### Key Capabilities Demonstrated:
1. **Zero-Wait Installation**: Install pre-compiled **`qwen-engine` v0.2.0** Linux wheels in 2 seconds directly from GitHub Releases (no C++/CUDA compiler required for Python inference!).
2. **Hugging Face Hub Integration**: Seamlessly fetch published production models from [`areen09/qwen-engine-models-qwenbin`](https://huggingface.co/areen09/qwen-engine-models-qwenbin).
3. **Single-GPU Streaming**: Real-time token streaming with exact Time To First Token (TTFT) and decode tokens/sec metrics on `cuda:0`.
4. **Hybrid Gated DeltaNet**: 75% KV cache memory reduction via linear-attention state recurrence ($S_t = S_{t-1} + eta_t (v_t - S_{t-1} k_t) k_t^T$).
5. **Dual Tesla T4 Tensor Parallelism ($TP = 2$)**: Seamlessly shard large 27B model layers across `cuda:0` and `cuda:1` with peer-to-peer (P2P) communication, fitting comfortably inside 16 GB VRAM with ~9 GB headroom.
6. **Deterministic C++ Benchmarks**: Run low-overhead C++ CLI benchmarks (`cli_bench`) to measure sustained throughput and latency across multiple warmup and measurement runs.

## Step 1: Kaggle Multi-GPU Diagnostics & Hardware Verification

Let's inspect the hardware environment, verify both Tesla T4 GPUs, check Peer-to-Peer (P2P) access between GPU 0 and GPU 1, and check available VRAM.

In [ ]:
import os
import sys
import torch

print(f"Python Version: {sys.version}")
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available:  {torch.cuda.is_available()}")

if torch.cuda.is_available():
    num_devices = torch.cuda.device_count()
    print(f"Detected CUDA Devices: {num_devices}\n")
    for d in range(num_devices):
        props = torch.cuda.get_device_properties(d)
        free_mem, total_mem = torch.cuda.mem_get_info(d)
        print(f"--- GPU {d}: {props.name} ---")
        print(f"  Compute Capability: sm_{props.major}{props.minor}")
        print(f"  Total VRAM:         {total_mem / (1024**3):.2f} GB")
        print(f"  Free VRAM:          {free_mem / (1024**3):.2f} GB")
        print(f"  Multi-Processors:   {props.multi_processor_count}")

    if num_devices >= 2:
        can_p2p = torch.cuda.can_device_access_peer(0, 1) and torch.cuda.can_device_access_peer(1, 0)
        print(f"\n✓ Dual GPU Peer-to-Peer (P2P) Access: {'ENABLED / SUPPORTED' if can_p2p else 'VIA PCIE HOST/BAR1'}")
    else:
        print("\n[Notice] Single GPU detected. Dual GPU TP=2 benchmarks require Kaggle 'GPU T4 x2' accelerator setting.")
else:
    raise RuntimeError("CUDA is not available. Please enable 'GPU T4 x2' in Kaggle Notebook Settings (right sidebar).")

## Step 2: Install Pre-Compiled Release Wheels (`v0.2.0`)

Instead of waiting 10+ minutes for a local C++/CUDA build from scratch, we install the **official pre-compiled release wheel** built directly with NVIDIA CUDA Toolkit 12.4 for Turing `sm_75` and Linux `manylinux_2_28`.

The wheel installs in **under 3 seconds** and includes all native Turing Tensor Core GEMV kernels, hybrid DeltaNet state operators, and Python bindings.

In [ ]:
import sys
import subprocess
import shutil
from pathlib import Path

# Detect Python ABI tag (e.g., cp310, cp311, cp312)
py_tag = f"cp{sys.version_info.major}{sys.version_info.minor}"
release_tag = "v0.2.0"
wheel_filename = f"qwen_engine-0.2.0-{py_tag}-{py_tag}-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl"
wheel_url = f"https://github.com/Areen-09/qwen-engine/releases/download/{release_tag}/{wheel_filename}"

print(f"[Engine Setup] Installing pre-compiled release wheel for {py_tag}...")
print(f"URL: {wheel_url}\n")

try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--upgrade", "--no-cache-dir", wheel_url])
    print("\n✓ Successfully installed qwen-engine from GitHub Release wheel!")
except Exception as e:
    print(f"Release wheel install failed: {e}")
    print("Falling back to local wheel build from source...")
    !pip install -q scikit-build-core nanobind ninja
    !git clone https://github.com/Areen-09/qwen-engine.git /tmp/qwen_src || git -C /tmp/qwen_src pull
    !pip install --no-build-isolation /tmp/qwen_src

# Verify import
import qwen_engine
print(f"\n✓ Verified qwen_engine module import! Version: {getattr(qwen_engine, '__version__', '0.2.0')}")
print(f"Available exports: {[attr for attr in dir(qwen_engine) if not attr.startswith('__')]}")

## Step 3: (Optional) Build Pure C++ Deterministic Benchmark (`cli_bench`)

If you want to measure raw bare-metal C++ latency without Python runtime overhead, we can build the lightweight `cli_bench` executable using Ninja.

With our cross-platform updates, targeting `--target cli_bench` compiles in **under 60 seconds** on Kaggle.

In [ ]:
BUILD_CLI_BENCH = True

if BUILD_CLI_BENCH:
    print("[CLI Build] Checking build tools and source repository...")
    !apt-get update -qq && apt-get install -y -qq cmake ninja-build build-essential
    
    qwen_dir = Path("/kaggle/working/qwen-engine") if os.path.exists("/kaggle") else Path("./qwen-engine")
    if not (qwen_dir / "src").exists():
        print("[CLI Build] Cloning qwen-engine repository...")
        !git clone https://github.com/Areen-09/qwen-engine.git {qwen_dir}
    else:
        print("[CLI Build] Updating local repository to latest main...")
        !cd {qwen_dir} && git fetch origin main && git reset --hard origin/main

    print("[CLI Build] Compiling cli_bench with nvcc for Turing sm_75...")
    !cd {qwen_dir} && cmake -B build -G Ninja -DCMAKE_BUILD_TYPE=Release -DCMAKE_CUDA_ARCHITECTURES=75 -DBUILD_PYTHON_BINDINGS=OFF
    !cd {qwen_dir} && cmake --build build --target cli_bench -j$(nproc)
    
    cli_bin = qwen_dir / "build" / "cli_bench"
    if cli_bin.exists():
        print(f"\n✓ Built CLI benchmark executable: {cli_bin}")
    else:
        print("[Notice] cli_bench build completed; binary located in build directory.")

## Step 4: Download Published Models from Hugging Face Hub

We download the published `.qwenbin` v2 artifacts directly from your Hugging Face repository:
[`areen09/qwen-engine-models-qwenbin`](https://huggingface.co/areen09/qwen-engine-models-qwenbin).

* **`qwen2.5-1.5b-w4a16.qwenbin`** (796 MB): Fast download in ~5-10 seconds.
* **`tokenizer.bin`** (3.7 MB): Pre-tokenized vocab and BPE merges.
* **`qwen3.5-4b-w4a16.qwenbin`** (~2.4 GB): 32-layer Hybrid DeltaNet.
* **`qwen3.8-27b-tp2-rank*.qwenbin`** (~7.1 GB each): Dual-GPU sharded weights.

In [ ]:
from huggingface_hub import hf_hub_download
from pathlib import Path
import os

HF_REPO_ID = "areen09/qwen-engine-models-qwenbin"
MODELS_DIR = Path("/kaggle/working/models") if os.path.exists("/kaggle") else Path("./models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Target Hugging Face Hub Repo: https://huggingface.co/{HF_REPO_ID}")
print(f"Local Storage Directory:     {MODELS_DIR}\n")

# 1. Always download 1.5B (796 MB) + Tokenizer (3.7 MB) for immediate testing
print("[1/4] Downloading Qwen 2.5-1.5B W4A16 (796 MB)...")
model_15b_path = hf_hub_download(repo_id=HF_REPO_ID, filename="qwen2.5-1.5b-w4a16.qwenbin", local_dir=str(MODELS_DIR))
print(f"  ✓ Downloaded: {model_15b_path} ({os.path.getsize(model_15b_path) / (1024**2):.1f} MB)")

print("[2/4] Downloading Tokenizer (3.7 MB)...")
tokenizer_path = hf_hub_download(repo_id=HF_REPO_ID, filename="tokenizer.bin", local_dir=str(MODELS_DIR))
print(f"  ✓ Downloaded: {tokenizer_path} ({os.path.getsize(tokenizer_path) / (1024**2):.1f} MB)")

# 2. Selectively download 4B or 27B on demand
DOWNLOAD_4B_HYBRID = True   # Set to True to test Qwen 3.5-4B (~2.4 GB)
DOWNLOAD_27B_TP2   = False  # Set to True to test Qwen 3.8-27B Dual T4 ranks (~14.2 GB total)

model_4b_path = None
if DOWNLOAD_4B_HYBRID:
    print("\n[3/4] Downloading Qwen 3.5-4B Hybrid DeltaNet (~2.4 GB)...")
    try:
        model_4b_path = hf_hub_download(repo_id=HF_REPO_ID, filename="qwen3.5-4b-w4a16.qwenbin", local_dir=str(MODELS_DIR))
        print(f"  ✓ Downloaded: {model_4b_path} ({os.path.getsize(model_4b_path) / (1024**2):.1f} MB)")
    except Exception as e:
        print(f"  [Notice] Could not download 4B model: {e}")

rank0_path, rank1_path = None, None
if DOWNLOAD_27B_TP2:
    print("\n[4/4] Downloading Qwen 3.8-27B 2-Way Tensor Parallel Ranks (~7.1 GB each)...")
    try:
        rank0_path = hf_hub_download(repo_id=HF_REPO_ID, filename="qwen3.8-27b-tp2-rank0-w4a16.qwenbin", local_dir=str(MODELS_DIR))
        print(f"  ✓ Downloaded Rank 0: {rank0_path} ({os.path.getsize(rank0_path) / (1024**3):.2f} GB)")
        rank1_path = hf_hub_download(repo_id=HF_REPO_ID, filename="qwen3.8-27b-tp2-rank1-w4a16.qwenbin", local_dir=str(MODELS_DIR))
        print(f"  ✓ Downloaded Rank 1: {rank1_path} ({os.path.getsize(rank1_path) / (1024**3):.2f} GB)")
    except Exception as e:
        print(f"  [Notice] Could not download 27B ranks: {e}")

## Step 5: Interactive Real-Time Streaming Inference (Tesla T4 `cuda:0`)

Let's run live streaming text generation on **Tesla T4 #0** using the Python API.
We measure:
* **Time To First Token (TTFT)** in milliseconds
* **Decode Throughput** in tokens per second
* **VRAM Allocation** vs GPU capacity

In [ ]:
import time
from qwen_engine import Engine

print("="*70)
print("  Qwen 2.5-1.5B (W4A16 Group-128) Streaming Inference on Tesla T4")
print("="*70)

# Instantiate engine on GPU 0
engine = Engine(model_path=model_15b_path, tokenizer_path=tokenizer_path, device_id=0)

prompt = "Explain why the sky is blue and how Rayleigh scattering works in three clear bullet points."
print(f"\nPrompt:\n\"{prompt}\"\n")
print("Streaming Response:")
print("-" * 50)

t0 = time.perf_counter()
first_token_time = None
token_count = 0

for piece in engine.stream_generate(prompt, max_new_tokens=150, temperature=0.7):
    if first_token_time is None:
        first_token_time = time.perf_counter()
    print(piece, end="", flush=True)
    token_count += 1

t1 = time.perf_counter()
print("\n" + "-" * 50)

ttft_ms = (first_token_time - t0) * 1000.0 if first_token_time else 0.0
decode_time_s = t1 - first_token_time if first_token_time else 1e-6
decode_tps = (token_count - 1) / decode_time_s if decode_time_s > 0 else 0.0

print(f"\nInference Performance Metrics:")
print(f"  Total Tokens Generated: {token_count}")
print(f"  Time to First Token:   {ttft_ms:.2f} ms")
print(f"  Decode Throughput:     {decode_tps:.1f} tokens/sec")
print(f"  End-to-End Latency:    {(t1 - t0):.2f} s")

# Memory Statistics
if hasattr(engine, "get_memory_stats"):
    stats = engine.get_memory_stats()
    print(f"\nVRAM Allocation Footprint:")
    print(f"  Allocated VRAM:        {stats.total_allocated_bytes / (1024**2):.1f} MB")
    print(f"  Total GPU Capacity:    {stats.device_total_bytes / (1024**2):.1f} MB")
    print(f"  VRAM Occupancy:        {(stats.total_allocated_bytes / stats.device_total_bytes)*100.0:.2f}%")

## Step 6: Hybrid Gated DeltaNet Inference (Qwen 3.5-4B)

The Qwen 3.5 architecture uses a **3:1 hybrid stacking ratio** (24 DeltaNet linear-attention layers, 8 standard GQA transformer layers).

Instead of caching all historical keys and values in memory (which scales quadratically $O(N)$ with context length), the DeltaNet layers maintain a fixed-size state matrix:
$$S_t = S_{t-1} + eta_t (v_t - S_{t-1} k_t) k_t^T$$

This provides **75% KV cache memory reduction**, allowing long contexts to run in minimal VRAM on a single 16 GB Tesla T4!

In [ ]:
if model_4b_path and os.path.exists(model_4b_path):
    print("="*70)
    print("  Qwen 3.5-4B Hybrid DeltaNet Streaming Inference")
    print("="*70)
    
    engine_4b = Engine(model_path=model_4b_path, tokenizer_path=tokenizer_path, device_id=0)
    
    prompt = "Describe the mathematical difference between standard softmax attention and linear recurrent state update."
    print(f"\nPrompt:\n\"{prompt}\"\n")
    print("Streaming Response:")
    print("-" * 50)
    
    t0 = time.perf_counter()
    tokens = 0
    for piece in engine_4b.stream_generate(prompt, max_new_tokens=100, temperature=0.3):
        print(piece, end="", flush=True)
        tokens += 1
    t1 = time.perf_counter()
    print("\n" + "-" * 50)
    print(f"Generated {tokens} tokens in {t1 - t0:.2f}s ({(tokens / (t1 - t0)):.1f} tok/s)")
else:
    print("[Step 6 Info] Qwen 3.5-4B was not downloaded in this run.")
    print("To test the 32-layer Hybrid DeltaNet, set DOWNLOAD_4B_HYBRID = True in Step 4.")

## Step 7: Dual Tesla T4 2-Way Tensor Parallelism ($TP = 2$)

For the large 27B model (Qwen 3.8-27B), single-GPU execution would exceed 16 GB VRAM.
With 2-Way Tensor Parallelism ($TP = 2$):
* **Rank 0** is hosted on **Tesla T4 #0** (`cuda:0`) (~7.1 GB VRAM)
* **Rank 1** is hosted on **Tesla T4 #1** (`cuda:1`) (~7.1 GB VRAM)
* Intermediate representations are synchronized across PCIe via peer-to-peer (P2P) AllReduce.
* Leaves **~8.9 GB VRAM headroom** on both GPUs for activations and KV cache!

In [ ]:
if torch.cuda.device_count() >= 2 and rank0_path and rank1_path:
    print("="*70)
    print("  Qwen 3.8-27B Dual Tesla T4 2-Way Tensor Parallel Inference (TP=2)")
    print("="*70)
    
    # Instantiate 2-way Tensor Parallel Engine across cuda:0 and cuda:1
    engine_27b = Engine(model_path=rank0_path, tokenizer_path=tokenizer_path, device_ids=[0, 1])
    
    prompt = "Provide a high-level summary of the architectural innovations in hybrid linear transformers."
    print(f"\nPrompt:\n\"{prompt}\"\n")
    print("Streaming Response:")
    print("-" * 50)
    
    t0 = time.perf_counter()
    tokens = 0
    for piece in engine_27b.stream_generate(prompt, max_new_tokens=100, temperature=0.7):
        print(piece, end="", flush=True)
        tokens += 1
    t1 = time.perf_counter()
    print("\n" + "-" * 50)
    print(f"Dual-GPU TP=2 Generated {tokens} tokens in {t1 - t0:.2f}s ({(tokens / (t1 - t0)):.1f} tok/s)")
elif torch.cuda.device_count() >= 2:
    print("="*70)
    print("  Dual Tesla T4 Hardware Verified (cuda:0 & cuda:1)")
    print("="*70)
    print("To run the live 27B Dual-GPU Tensor Parallel benchmark:")
    print("1. Set DOWNLOAD_27B_TP2 = True in Step 4 to download both sharded ranks (~7.1 GB each).")
    print(f"2. Hub Repository: https://huggingface.co/{HF_REPO_ID}")
    print("3. Alternatively, run the CLI benchmark below on both devices!")
else:
    print("[Notice] Dual-GPU TP=2 requires 2 GPUs. Running in single-GPU mode.")

## Step 8: Deterministic C++ CLI Benchmarking (`cli_bench`)

We execute low-overhead C++ CLI benchmarks using synthetic deterministic tokens (128 prompt tokens, 256 generated tokens, multiple warmups and runs).
This isolates pure GPU kernel efficiency and eliminates all host Python runtime jitter.

In [ ]:
# Find cli_bench executable
cli_candidates = [
    "/kaggle/working/qwen-engine/build/cli_bench",
    "./qwen-engine/build/cli_bench",
    "./build/cli_bench",
    "./build/bin/cli_bench"
]

cli_bin = None
for cand in cli_candidates:
    if os.path.exists(cand):
        cli_bin = cand
        break

if cli_bin and os.path.exists(model_15b_path):
    print("="*70)
    print("  [Benchmark 1] Single-GPU Deterministic C++ Benchmark (Tesla T4 cuda:0)")
    print("="*70)
    !{cli_bin} --model {model_15b_path} --device 0 --prompt-len 128 --gen-len 128 --warmup 5 --runs 10
else:
    print("[CLI Notice] cli_bench binary or model not available for CLI bench test.")

if cli_bin and torch.cuda.device_count() >= 2:
    print("\n" + "="*70)
    print("  [Benchmark 2] Dual-GPU Tensor Parallel C++ Benchmark (cuda:0 & cuda:1)")
    print("="*70)
    tp_model = rank0_path if (rank0_path and os.path.exists(rank0_path)) else model_15b_path
    !{cli_bin} --model {tp_model} --devices 0,1 --prompt-len 64 --gen-len 64 --warmup 3 --runs 5

## Step 9: Performance Summary & Deployment Architecture

### Architecture & Memory Characteristics on Kaggle Dual Tesla T4s:

| Feature | Qwen 2.5-1.5B | Qwen 3.5-4B | Qwen 3.8-27B ($TP = 2$) |
| :--- | :--- | :--- | :--- |
| **Model Size (W4A16)** | **796 MB** | **~2.4 GB** | **~14.2 GB (~7.1 GB / GPU)** |
| **Layer Architecture** | 28 GQA Layers | 24 DeltaNet + 8 GQA | 48 DeltaNet + 16 GQA |
| **Target GPU Setup** | 1x Tesla T4 (`cuda:0`) | 1x Tesla T4 (`cuda:0`) | **2x Tesla T4 (`cuda:0` + `cuda:1`)** |
| **VRAM Headroom** | ~15.0 GB | ~13.5 GB | **~8.9 GB per GPU** |
| **Quantization Scheme** | W4A16 Group-128 | W4A16 Group-128 | W4A16 Group-128 |
| **Kernel Implementation**| Turing Tensor Core GEMV | Hybrid State Recurrence | Column/Row Sharded GEMV + AllReduce |

---

### Integrating Into Your Local Application:
You can use `qwen_engine` in your local Python scripts, microservices, or FastAPI backends:
```python
from qwen_engine import Engine

# Single GPU
engine = Engine.from_pretrained("areen09/qwen-engine-models-qwenbin", device_id=0)
for chunk in engine.stream_generate("Hello world!"):
    print(chunk, end="", flush=True)

# Dual GPU TP=2
engine_tp = Engine.from_pretrained("areen09/qwen-engine-models-qwenbin", device_ids=[0, 1])
```